# Audio Classification

> Labelling sound - tagging, keyword spotting, emotion, zero-shot: how the models work, the mid-2026 landscape, evaluation, and runnable transformers code on GPU or CPU.

- skip_showdoc: true
- skip_exec: true

## 1. What is Audio Classification?

Audio classification maps an audio clip to one or more **labels**. Common variants:

- **Sound event tagging** - what sounds are present (AudioSet's 527 classes, ESC-50).
- **Keyword spotting (KWS)** - detect command words ("yes", "stop").
- **Speech emotion recognition** - happy / sad / angry / neutral.
- **Speaker identification** and **language ID**.
- **Music genre / auto-tagging**, **acoustic scene classification**.

**Output.** A single label (multi-class) or many labels with scores (multi-label tagging).

**Neighbouring tasks:** Voice Activity Detection (a binary special case), ASR, and audio captioning.

---

## 2. How Modern Audio Classification Works

1. **Hand-crafted features + GMM/SVM (pre-2017).** MFCCs into a shallow classifier. Legacy.
2. **CNN on log-mel (VGGish, PANNs, 2019).** Treat the spectrogram as an image; large-scale AudioSet pretraining transfers well.
3. **Audio Spectrogram Transformer (AST, 2021).** A ViT over spectrogram patches - AudioSet state of the art and the default tagging backbone.
4. **Self-supervised encoders (wav2vec 2.0, HuBERT, WavLM, 2021).** Pretrain on unlabeled speech, fine-tune a light head for KWS, emotion, speaker ID; **BEATs** (2022) leads general audio tagging.
5. **CLAP (2023).** Contrastive language-audio pretraining enables **zero-shot** classification: score the clip against arbitrary text label prompts, no fine-tuning. By 2025-2026 CLAP-style zero-shot and unified audio encoders are the flexible default.

---

## 3. Evaluation Metrics

- **Accuracy / F1.** For single-label tasks (KWS, emotion, ESC-50).
- **mAP (mean Average Precision).** The standard for **multi-label** tagging (AudioSet) - averages precision across recall for each class.
- **d-prime / AUC.** Alternative multi-label summaries.
- **Confusion matrix.** Where the errors go - shown as an ECharts heatmap in the benchmark.

The cell computes accuracy and macro-F1 with `scikit-learn` on toy predictions.

---

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

y_true = ["dog", "rain", "dog", "engine", "rain"]
y_pred = ["dog", "rain", "cat", "engine", "rain"]
print("accuracy:", accuracy_score(y_true, y_pred))
print("macro-F1:", f1_score(y_true, y_pred, average="macro"))

## 4. The Model Landscape (mid-2026)

| Model | Params | License | Task | Best for |
|-------|--------|---------|------|----------|
| MIT/ast-finetuned-audioset-10-10-0.4593 | 86M | BSD-3 | AudioSet tagging (527) | general sound tagging |
| superb/hubert-large-superb-er | 300M | Apache 2.0 | speech emotion | emotion recognition |
| superb/wav2vec2-base-superb-ks | 95M | Apache 2.0 | keyword spotting (35) | command words |
| laion/clap-htsat-unfused | 150M | Apache 2.0 | zero-shot | label from text prompts |
| MIT/ast-finetuned-speech-commands-v2 | 86M | BSD-3 | keyword spotting | speech commands |

Benchmarks: **SUPERB** (speech tasks), **AudioSet mAP** (tagging), **HEAR** (holistic). All of these load through the `transformers` `audio-classification` (or `zero-shot-audio-classification`) pipeline.

---

## 5. Setup

Package roles:

- `transformers` (>=5.13) + `torch` - AST, HuBERT, wav2vec2, CLAP
- `datasets` - ESC-50 environmental-sound clips with labels
- `scikit-learn` - accuracy / F1 / confusion matrix
- `pyecharts` - accuracy bar + confusion-matrix heatmap

---

In [ ]:
import gc
import time
import urllib.request
from pathlib import Path

import torch
from dotenv import find_dotenv, load_dotenv

# Knowledge/.env sets HF_TOKEN - authenticated HF Hub requests get higher rate limits
load_dotenv(find_dotenv(usecwd=True))

device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device != "cpu":
    print(torch.cuda.get_device_name(0))
print("device:", device)


def vram(tag=""):
    "Report current GPU memory (allocated / reserved). No-op on CPU."
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"VRAM {tag:16s} {alloc:5.2f} GB allocated / {reserved:5.2f} GB reserved")


def free_memory():
    "GC then release cached CPU/GPU memory. Call right after `del model`.\n\n    `del` drops the Python reference; this reclaims the RAM and hands the\n    freed VRAM back to the CUDA allocator so usage stays flat across cells.\n    "
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


# All downloads (samples, HF cache) go to DL_tasks/datasets/ (gitignored)
DATA_DIR = Path("../../datasets")
DATA_DIR.mkdir(exist_ok=True)

import numpy as np
from datasets import load_dataset

# ESC-50: 2000 labelled 5 s environmental-sound clips, 50 categories
esc = load_dataset("ashraq/esc50", split="train", cache_dir=str(DATA_DIR / "hf_cache"))
CATEGORIES = sorted(set(esc["category"]))
print(len(esc), "clips,", len(CATEGORIES), "categories")


def esc_clip(row):
    "Return {array, sampling_rate} for a transformers audio pipeline."
    a = row["audio"]
    return {"array": np.asarray(a["array"], dtype="float32"), "sampling_rate": a["sampling_rate"]}

## 6. AST - AudioSet tagging

The Audio Spectrogram Transformer tags a clip against AudioSet's 527 classes. The `audio-classification` pipeline handles resampling to 16 kHz and returns scored labels; `top_k` sets how many.

---

In [ ]:
from transformers import pipeline

tagger = pipeline("audio-classification", model="MIT/ast-finetuned-audioset-10-10-0.4593",
                  device=device, top_k=5)
clip = esc_clip(esc[0])
t0 = time.perf_counter()
print(f"true category: {esc[0]['category']}")
for p in tagger(clip):
    print(f"  {p['score']:.3f}  {p['label']}")
print(f"{time.perf_counter() - t0:.2f}s")

del tagger
free_memory()
vram("after ast")

## 7. CLAP - zero-shot classification

CLAP scores the clip against **arbitrary text prompts**, so it classifies ESC-50 with no fine-tuning - we just pass the 50 category names as candidate labels. This is the flexible modern default.

---

In [ ]:
zsc = pipeline("zero-shot-audio-classification", model="laion/clap-htsat-unfused", device=device)
prompts = [f"the sound of {c.replace('_', ' ')}" for c in CATEGORIES]

clip = esc_clip(esc[0])
t0 = time.perf_counter()
preds = zsc(clip["array"], candidate_labels=prompts)
print(f"true: {esc[0]['category']}  ->  top: {preds[0]['label']}  ({preds[0]['score']:.3f})")
print(f"{time.perf_counter() - t0:.2f}s")

del zsc
free_memory()
vram("after clap")

## 8. Head-to-head Benchmark

Compare **CLAP zero-shot** against a **fused CLAP** variant on an ESC-50 subset: **accuracy** + **RTF**, plus a confusion-matrix heatmap for the zero-shot model. Same clips, same label set. A 50-clip subset is a smoke test; ESC-50's official 5-fold protocol is the real evaluation.

---

In [ ]:
# ECharts (pyecharts) is the repo standard for all charts - it renders interactive
# and embeds straight into the Quarto docs via .render_notebook().
from pyecharts import options as opts
from pyecharts.charts import Bar


def bar_chart(title, categories, series, y_name=""):
    "Grouped bar chart. `series` is a dict {name: [values aligned to categories]}."
    chart = Bar(init_opts=opts.InitOpts(width="720px", height="420px"))
    chart.add_xaxis([str(c) for c in categories])
    for name, vals in series.items():
        chart.add_yaxis(name, [round(float(v), 4) for v in vals])
    chart.set_global_opts(
        title_opts=opts.TitleOpts(title=title),
        yaxis_opts=opts.AxisOpts(name=y_name),
        xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=20)),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
    return chart.render_notebook()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

N = 50  # clips to evaluate
subset = esc.select(range(N))
refs = [row["category"] for row in subset]
prompts = [f"the sound of {c.replace('_', ' ')}" for c in CATEGORIES]

results, preds_for_cm = {}, None
for model_id in ["laion/clap-htsat-unfused", "laion/clap-htsat-fused"]:
    zsc = pipeline("zero-shot-audio-classification", model=model_id, device=device)
    t0 = time.perf_counter()
    hyps = []
    for row in subset:
        p = zsc(esc_clip(row)["array"], candidate_labels=prompts)
        top = p[0]["label"]
        hyps.append(CATEGORIES[prompts.index(top)])
    elapsed = time.perf_counter() - t0
    acc = accuracy_score(refs, hyps)
    results[model_id.split("/")[-1]] = {"acc": acc, "rtf": elapsed / (N * 5.0)}
    print(f"{model_id:30s} acc {acc:6.2%}  {elapsed:5.1f}s")
    if preds_for_cm is None:
        preds_for_cm = hyps
    del zsc
    free_memory()
vram("after benchmark")

In [ ]:
names = list(results)
bar_chart("Audio classification: zero-shot accuracy on ESC-50 subset",
          names, {"accuracy": [results[n]["acc"] for n in names]}, y_name="accuracy")

In [ ]:
# Confusion matrix as an ECharts heatmap (categories that actually appear in the subset)
from pyecharts import options as opts
from pyecharts.charts import HeatMap

present = sorted(set(refs))
cm = confusion_matrix(refs, preds_for_cm, labels=present)
data = [[j, i, int(cm[i][j])] for i in range(len(present)) for j in range(len(present))]

hm = HeatMap(init_opts=opts.InitOpts(width="760px", height="620px"))
hm.add_xaxis(present)
hm.add_yaxis("true vs predicted", present, data,
             label_opts=opts.LabelOpts(is_show=True, position="inside"))
hm.set_global_opts(
    title_opts=opts.TitleOpts(title="ESC-50 confusion (CLAP unfused)"),
    xaxis_opts=opts.AxisOpts(axislabel_opts=opts.LabelOpts(rotate=45)),
    visualmap_opts=opts.VisualMapOpts(max_=int(cm.max()), orient="vertical", pos_left="right"),
)
hm.render_notebook()

## 9. Going Further

- **Multi-label tagging at scale.** Evaluate AST with **mAP** on the full AudioSet eval set; **BEATs** and CED push the state of the art.
- **Fine-tuning.** Fine-tune wav2vec2 / HuBERT for your own KWS, emotion, or speaker-ID head: [HF audio classification guide](https://huggingface.co/docs/transformers/tasks/audio_classification).
- **Speaker ID / diarization.** `speechbrain`'s ECAPA-TDNN embeddings for verification and clustering.
- **Zero-shot everywhere.** Swap the CLAP candidate prompts to classify any new taxonomy with no training data.

---